# Lesson 7.2: ANOVA: Comparing Three or More Means

Compare the exam scores of three teaching methods with a one-way ANOVA, then use Tukey's HSD to find which pairs of methods differ.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. Enter the three groups

Five students per teaching method, the same 15 scores as the lesson. We look at each group mean first, because ANOVA asks whether those means are further apart than the spread inside each group would explain.

In [1]:
import numpy as np
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

lecture = [78, 82, 85, 79, 76]
flipped = [88, 91, 87, 93, 85]
problem = [72, 68, 74, 70, 71]

for name, g in [('lecture', lecture), ('flipped', flipped), ('problem', problem)]:
    print(f'{name:8s} mean = {np.mean(g):.1f}   sd = {np.std(g, ddof=1):.2f}')

lecture  mean = 80.0   sd = 3.54
flipped  mean = 88.8   sd = 3.19
problem  mean = 71.0   sd = 2.24


## 2. Run the one-way ANOVA

f_oneway compares the spread between group means with the spread inside the groups. F near 1 means no difference; a large F with a small p-value says at least one mean differs.

In [2]:
res = stats.f_oneway(lecture, flipped, problem)

print(f'F       = {res.statistic:.2f}')
print('df      = (2, 12)')
print(f'p-value = {res.pvalue:.7f}')

F       = 42.90
df      = (2, 12)
p-value = 0.0000034


## 3. Measure the effect size

Eta squared is the share of all the score variation that is explained by teaching method. It is the between-group sum of squares divided by the total.

In [3]:
scores = np.array(lecture + flipped + problem)
grand = scores.mean()

ssb = sum(len(g) * (np.mean(g) - grand) ** 2 for g in (lecture, flipped, problem))
sst = ((scores - grand) ** 2).sum()

print(f'between SS = {ssb:.1f}, total SS = {sst:.1f}')
print(f'eta squared = {ssb / sst:.3f}')

between SS = 792.1, total SS = 902.9
eta squared = 0.877


## 4. Find which pairs differ with Tukey

A significant ANOVA only says some mean is different. Tukey's HSD compares every pair while keeping the overall false-alarm rate at 5%. In the table, reject = True means that pair differs.

In [4]:
labels = ['lecture'] * 5 + ['flipped'] * 5 + ['problem'] * 5

tukey = pairwise_tukeyhsd(scores, labels, alpha=0.05)

print(tukey.summary())

  Multiple Comparison of Means - Tukey HSD, FWER=0.05   
 group1  group2 meandiff p-adj   lower    upper   reject
--------------------------------------------------------
flipped lecture     -8.8 0.0017 -13.9271  -3.6729   True
flipped problem    -17.8    0.0 -22.9271 -12.6729   True
lecture problem     -9.0 0.0014 -14.1271  -3.8729   True
--------------------------------------------------------


## Try this

- Make the three problem-based scores 80, 76, 82, 78, 79 (closer to the others) and predict what happens to F and to the Tukey table.
- Change one flipped score to 60. How do the group mean, F and the p-value respond?

## The same idea in R, SPSS and Stata

**R**

```r
score <- c(78, 82, 85, 79, 76, 88, 91, 87, 93, 85, 72, 68, 74, 70, 71)
method <- factor(rep(c("lecture", "flipped", "problem"), each = 5))
fit <- aov(score ~ method)
summary(fit)
TukeyHSD(fit)
```

**SPSS** (Analyze > Compare Means > One-Way ANOVA (Post Hoc: Tukey))

```
DATA LIST FREE / method score.
BEGIN DATA
1 78  1 82  1 85  1 79  1 76
2 88  2 91  2 87  2 93  2 85
3 72  3 68  3 74  3 70  3 71
END DATA.
ONEWAY score BY method
  /POSTHOC=TUKEY ALPHA(0.05).
```

**Stata**

```
clear
set obs 15
generate score = real(word("78 82 85 79 76 88 91 87 93 85 72 68 74 70 71", _n))
generate method = ceil(_n / 5)
oneway score method
quietly anova score method
pwcompare method, mcompare(tukey) effects
```